# Credit Card Fraud Detection System
## 23AIML076

**Objective:** Design and build a robust, real-world fraud detection system using both Machine Learning models and an Artificial Neural Network (ANN).

**Dataset:** Credit Card Fraud Detection dataset from Kaggle

**Reference:** Inspired by best-techniques-and-metrics-for-imbalanced-dataset notebook

---
# Task 1: Data Understanding & Cleaning

In this section, we explore the dataset structure, data types, missing values, and perform necessary cleaning and preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

print("Libraries imported successfully!")

### 1.1 Loading the Dataset

In [ ]:
# Load training and test datasets
train_df = pd.read_csv('fraudTrain.csv/fraudTrain.csv')
test_df = pd.read_csv('fraudTest.csv/fraudTest.csv')

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"\nTotal records: {train_df.shape[0] + test_df.shape[0]}")

### 1.2 Exploring Dataset Structure

In [ ]:
# Display first few rows
print("=== Training Data - First 5 Rows ===")
train_df.head()

In [ ]:
# Data types and info
print("=== Training Data Info ===")
train_df.info()

In [ ]:
# Statistical summary
print("=== Statistical Summary (Numerical Features) ===")
train_df.describe()

In [ ]:
# Statistical summary for categorical features
print("=== Statistical Summary (Categorical Features) ===")
train_df.describe(include='object')

### 1.3 Checking Missing Values

In [ ]:
# Check for missing values
print("=== Missing Values in Training Set ===")
missing_train = train_df.isnull().sum()
print(missing_train[missing_train > 0] if missing_train.sum() > 0 else "No missing values found!")

print("\n=== Missing Values in Test Set ===")
missing_test = test_df.isnull().sum()
print(missing_test[missing_test > 0] if missing_test.sum() > 0 else "No missing values found!")

### 1.4 Checking Duplicates

In [ ]:
# Check for duplicate rows
print(f"Duplicate rows in training set: {train_df.duplicated().sum()}")
print(f"Duplicate rows in test set: {test_df.duplicated().sum()}")

### 1.5 Data Cleaning

We'll drop unnecessary columns and clean the data for modeling.

In [ ]:
# Drop the 'Unnamed: 0' column (just an index)
train_df.drop('Unnamed: 0', axis=1, inplace=True)
test_df.drop('Unnamed: 0', axis=1, inplace=True)

# Combine for consistent preprocessing
print("Columns after dropping index:")
print(train_df.columns.tolist())
print(f"\nTraining shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

---
# Task 2: Exploratory Data Analysis (EDA)

Let's perform detailed univariate and bivariate analysis to identify fraud patterns and key insights.

### 2.1 Target Variable Distribution (Class Imbalance Check)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
fraud_counts = train_df['is_fraud'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(fraud_counts.index.astype(str), fraud_counts.values, color=colors, edgecolor='black')
axes[0].set_xlabel('Class (0=Legit, 1=Fraud)')
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Class Distribution')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 500, str(v), ha='center', fontweight='bold')

# Percentage pie chart
labels = ['Legitimate', 'Fraudulent']
axes[1].pie(fraud_counts.values, labels=labels, colors=colors, autopct='%1.2f%%', 
            startangle=90, explode=(0, 0.1), shadow=True)
axes[1].set_title('Fraud vs Legitimate Transactions')

plt.tight_layout()
plt.show()

print(f"\nClass Distribution:")
print(f"Legitimate (0): {fraud_counts[0]} ({fraud_counts[0]/len(train_df)*100:.2f}%)")
print(f"Fraudulent (1): {fraud_counts[1]} ({fraud_counts[1]/len(train_df)*100:.2f}%)")
print(f"\nImbalance Ratio: 1:{fraud_counts[0]//fraud_counts[1]}")

**Observation:** The dataset is highly imbalanced! The ratio of legitimate to fraudulent transactions is approximately 1:170. This is a classic class imbalance problem that we need to handle carefully.

### 2.2 Univariate Analysis

In [ ]:
# Transaction Amount Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall amount distribution
axes[0].hist(train_df['amt'], bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Transaction Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Transaction Amounts')
axes[0].set_xlim(0, 1000)

# Log-scaled amount
axes[1].hist(np.log1p(train_df['amt']), bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Log(Transaction Amount)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed Amount Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Category distribution
fig, ax = plt.subplots(figsize=(14, 6))
cat_counts = train_df['category'].value_counts()
cat_counts.plot(kind='barh', color='steelblue', edgecolor='black', ax=ax)
ax.set_xlabel('Number of Transactions')
ax.set_ylabel('Category')
ax.set_title('Transaction Categories')
plt.tight_layout()
plt.show()

In [ ]:
# Gender distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
gender_counts = train_df['gender'].value_counts()
axes[0].bar(gender_counts.index, gender_counts.values, color=['#3498db', '#e91e63'], edgecolor='black')
axes[0].set_title('Gender Distribution')
axes[0].set_ylabel('Count')

# Age distribution (calculate from dob)
train_df['dob'] = pd.to_datetime(train_df['dob'])
train_df['trans_date_trans_time'] = pd.to_datetime(train_df['trans_date_trans_time'])
train_df['age'] = (train_df['trans_date_trans_time'] - train_df['dob']).dt.days // 365

axes[1].hist(train_df['age'], bins=50, color='mediumpurple', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Age Distribution of Cardholders')

plt.tight_layout()
plt.show()

### 2.3 Bivariate Analysis

In [ ]:
# Amount by Fraud Status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
train_df.boxplot(column='amt', by='is_fraud', ax=axes[0])
axes[0].set_title('Transaction Amount by Fraud Status')
axes[0].set_xlabel('Is Fraud (0=No, 1=Yes)')
axes[0].set_ylabel('Amount ($)')
plt.sca(axes[0])
plt.xticks([1, 2], ['Legitimate', 'Fraudulent'])

# Mean amount comparison
mean_amt = train_df.groupby('is_fraud')['amt'].mean()
axes[1].bar(['Legitimate', 'Fraudulent'], mean_amt.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[1].set_ylabel('Mean Amount ($)')
axes[1].set_title('Average Transaction Amount by Class')
for i, v in enumerate(mean_amt.values):
    axes[1].text(i, v + 2, f'${v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Mean amount - Legitimate: ${mean_amt[0]:.2f}")
print(f"Mean amount - Fraudulent: ${mean_amt[1]:.2f}")

In [ ]:
# Fraud rate by category
fig, ax = plt.subplots(figsize=(14, 6))
fraud_by_cat = train_df.groupby('category')['is_fraud'].mean().sort_values(ascending=True)
fraud_by_cat.plot(kind='barh', color='coral', edgecolor='black', ax=ax)
ax.set_xlabel('Fraud Rate')
ax.set_ylabel('Category')
ax.set_title('Fraud Rate by Transaction Category')
ax.axvline(x=train_df['is_fraud'].mean(), color='red', linestyle='--', label=f'Overall Fraud Rate: {train_df["is_fraud"].mean():.4f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fraud by Hour of Day
train_df['hour'] = train_df['trans_date_trans_time'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transaction count by hour
hour_counts = train_df.groupby(['hour', 'is_fraud']).size().unstack(fill_value=0)
hour_counts.plot(kind='bar', stacked=False, ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Count')
axes[0].set_title('Transactions by Hour')
axes[0].legend(['Legitimate', 'Fraudulent'])

# Fraud rate by hour
fraud_rate_hour = train_df.groupby('hour')['is_fraud'].mean()
axes[1].plot(fraud_rate_hour.index, fraud_rate_hour.values, 'ro-', linewidth=2, markersize=6)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate')
axes[1].set_title('Fraud Rate by Hour of Day')
axes[1].axhline(y=train_df['is_fraud'].mean(), color='blue', linestyle='--', alpha=0.5, label='Overall Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nKey Insight: Fraud transactions tend to occur more frequently during late night/early morning hours!")

In [ ]:
# Correlation heatmap (numerical features)
fig, ax = plt.subplots(figsize=(12, 8))
numerical_cols = train_df.select_dtypes(include=[np.number]).columns
corr_matrix = train_df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
plt.show()

### 2.4 Key EDA Insights

1. **Severe Class Imbalance:** Only ~0.58% of transactions are fraudulent - classic imbalanced dataset problem
2. **Amount Patterns:** Fraudulent transactions have significantly higher average amounts
3. **Time Patterns:** Fraud is more common during late night hours (22:00 - 04:00)
4. **Category Patterns:** Some categories like `shopping_net`, `grocery_pos` have higher fraud rates
5. **Geographic Patterns:** Transaction and merchant locations may provide useful distance features

---
# Task 3: Data Preprocessing

### Steps:
- Feature scaling
- Handle class imbalance (covered in detail in Task 5 & 6)
- Train-test split

### 3.1 Feature Engineering & Preparation

We need to transform raw features into model-ready format.

In [ ]:
# Also process test set
test_df['dob'] = pd.to_datetime(test_df['dob'])
test_df['trans_date_trans_time'] = pd.to_datetime(test_df['trans_date_trans_time'])
test_df['age'] = (test_df['trans_date_trans_time'] - test_df['dob']).dt.days // 365
test_df['hour'] = test_df['trans_date_trans_time'].dt.hour

# Feature Engineering for both datasets
def engineer_features(df):
    # Extract time features
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month'] = df['trans_date_trans_time'].dt.month
    
    # Distance between customer and merchant
    df['distance'] = np.sqrt((df['lat'] - df['merch_lat'])**2 + (df['long'] - df['merch_long'])**2)
    
    # Log of amount (reduces skewness)
    df['log_amt'] = np.log1p(df['amt'])
    
    return df

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

print("Engineered features added!")
print(f"New columns: ['age', 'hour', 'day_of_week', 'month', 'distance', 'log_amt']")

### 3.2 Selecting Features for Modeling

We drop columns that are not useful for modeling (like names, addresses, transaction IDs) and keep only relevant numerical features.

In [ ]:
# Select features for modeling
# Drop non-useful columns (personal info, IDs, etc.)
drop_cols = ['trans_date_trans_time', 'cc_num', 'merchant', 'first', 'last', 
             'street', 'city', 'state', 'zip', 'dob', 'trans_num', 'job']

# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

# Encode 'category' and 'gender'
le_cat = LabelEncoder()
le_gen = LabelEncoder()

train_df['category_encoded'] = le_cat.fit_transform(train_df['category'])
test_df['category_encoded'] = le_cat.transform(test_df['category'])

train_df['gender_encoded'] = le_gen.fit_transform(train_df['gender'])
test_df['gender_encoded'] = le_gen.transform(test_df['gender'])

# Features to use
feature_cols = ['amt', 'log_amt', 'lat', 'long', 'city_pop', 'unix_time', 
                'merch_lat', 'merch_long', 'age', 'hour', 'day_of_week', 
                'month', 'distance', 'category_encoded', 'gender_encoded']

X_train = train_df[feature_cols].copy()
y_train = train_df['is_fraud'].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df['is_fraud'].copy()

print(f"Training features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"\nSelected features: {feature_cols}")

### 3.3 Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for convenience
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)

print("Feature scaling complete!")
print(f"\nScaled training data statistics:")
print(X_train_scaled.describe().round(2))

### 3.4 Understanding Class Imbalance

This is critical! With ~0.58% fraud rate, naive models would just predict everything as legitimate and still get ~99.4% accuracy. We need proper handling.

In [ ]:
# Visualize class imbalance impact
print("=== Class Distribution ===")
print(f"Training set:")
print(f"  Legitimate: {(y_train==0).sum()} ({(y_train==0).mean()*100:.2f}%)")
print(f"  Fraudulent: {(y_train==1).sum()} ({(y_train==1).mean()*100:.2f}%)")
print(f"\nTest set:")
print(f"  Legitimate: {(y_test==0).sum()} ({(y_test==0).mean()*100:.2f}%)")
print(f"  Fraudulent: {(y_test==1).sum()} ({(y_test==1).mean()*100:.2f}%)")
print(f"\nImbalance ratio in training: 1:{(y_train==0).sum()//(y_train==1).sum()}")
print("\n[!] Accuracy alone is NOT a good metric for this problem!")
print("We need Precision, Recall, F1-Score, and ROC-AUC!")

---
# Task 4: Feature Engineering & Selection

### 4.1 Feature Engineering
We've already created several engineered features:
- **age**: Derived from date of birth
- **hour, day_of_week, month**: Temporal features from transaction time
- **distance**: Geographical distance between customer and merchant
- **log_amt**: Log-transformed transaction amount
- **category_encoded, gender_encoded**: Label-encoded categorical variables

### 4.2 Feature Selection using Mutual Information

Mutual Information measures how much information the presence/absence of a feature contributes to predicting the target variable. It's a good technique for both linear and non-linear relationships.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Calculate mutual information scores
# Use a sample for efficiency given the large dataset
sample_idx = np.random.choice(len(X_train_scaled), size=min(50000, len(X_train_scaled)), replace=False)
X_sample = X_train_scaled.iloc[sample_idx]
y_sample = y_train.iloc[sample_idx]

mi_scores = mutual_info_classif(X_sample, y_sample, random_state=42)
mi_df = pd.DataFrame({'Feature': feature_cols, 'MI_Score': mi_scores})
mi_df = mi_df.sort_values('MI_Score', ascending=False)

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mi_df['Feature'], mi_df['MI_Score'], color='steelblue', edgecolor='black')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Importance (Mutual Information)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nFeature Rankings by Mutual Information:")
for i, row in mi_df.iterrows():
    print(f"  {row['Feature']:>20s}: {row['MI_Score']:.4f}")

### 4.3 Feature Selection Justification

**Approach:** We use Mutual Information (MI) scores for feature selection because:
1. MI captures both linear and non-linear dependencies
2. It doesn't assume any specific distribution
3. It works well with both categorical and continuous features
4. It quantifies the amount of information each feature provides about the target

**Key findings:**
- `distance`, `amt`, `category_encoded`, and `hour` are among the most informative features
- All features show non-zero MI scores, so we retain all of them
- The engineered features (`distance`, `log_amt`, `age`, `hour`) contribute significantly

---
# Task 5: Model Building (Machine Learning)

We'll train at least 4 models, compare them, and perform hyperparameter tuning.

**Models:**
1. Logistic Regression
2. Random Forest
3. XGBoost
4. Decision Tree

**Strategy:** Since the dataset is very large and imbalanced, we use:
- SMOTE for handling class imbalance (on training data only)
- Stratified sampling for efficiency

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                              roc_auc_score, precision_recall_curve, 
                              roc_curve, f1_score, precision_score, recall_score)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import time

print("All model libraries imported!")

### 5.1 Handling Class Imbalance with SMOTE

SMOTE (Synthetic Minority Over-sampling Technique) creates synthetic samples of the minority class by interpolating between existing minority samples. This is done ONLY on the training data to avoid data leakage.

**Reference:** Following the best-techniques-and-metrics-for-imbalanced-dataset notebook approach.

In [ ]:
# Due to large dataset size, we'll use a stratified sample for training
# This makes training feasible while keeping class proportions
from sklearn.model_selection import train_test_split

# Use a manageable sample size (stratified)
# Taking ~100k samples to balance speed and quality
sample_size = 100000
if len(X_train_scaled) > sample_size:
    X_train_sample, _, y_train_sample, _ = train_test_split(
        X_train_scaled, y_train, train_size=sample_size, 
        stratify=y_train, random_state=42
    )
    print(f"Using stratified sample of {sample_size} for training")
else:
    X_train_sample = X_train_scaled.copy()
    y_train_sample = y_train.copy()

print(f"Sample size: {len(X_train_sample)}")
print(f"Fraud in sample: {y_train_sample.sum()} ({y_train_sample.mean()*100:.2f}%)")

# Apply SMOTE to training sample only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_sample, y_train_sample)

print(f"\nAfter SMOTE:")
print(f"  Total samples: {len(X_train_smote)}")
print(f"  Legitimate: {(y_train_smote==0).sum()}")
print(f"  Fraudulent: {(y_train_smote==1).sum()}")
print(f"  Ratio: 1:1 (Balanced!)")

In [ ]:
# Visualize before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before SMOTE
before = y_train_sample.value_counts()
axes[0].bar(['Legit', 'Fraud'], before.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Before SMOTE')
axes[0].set_ylabel('Count')
for i, v in enumerate(before.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# After SMOTE
after = pd.Series(y_train_smote).value_counts()
axes[1].bar(['Legit', 'Fraud'], after.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[1].set_title('After SMOTE')
axes[1].set_ylabel('Count')
for i, v in enumerate(after.values):
    axes[1].text(i, v + 100, str(v), ha='center', fontweight='bold')

plt.suptitle('Class Distribution: Before vs After SMOTE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Training Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, 
                                             max_depth=10, n_jobs=-1),
}

# Try importing XGBoost
try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(n_estimators=100, max_depth=6, 
                                       learning_rate=0.1, random_state=42,
                                       use_label_encoder=False, eval_metric='logloss')
    print("XGBoost available!")
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    models['Gradient Boosting'] = GradientBoostingClassifier(n_estimators=100, 
                                                              max_depth=6, random_state=42)
    print("Using Gradient Boosting as XGBoost alternative")

# Train and evaluate each model
results = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print('='*50)
    
    start = time.time()
    model.fit(X_train_smote, y_train_smote)
    train_time = time.time() - start
    
    # Predict on test set
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'train_time': train_time,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"  Training Time: {train_time:.2f}s")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent'])}")

### 5.3 Model Comparison

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    name: {
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1-Score': r['f1'],
        'ROC-AUC': r['roc_auc'],
        'Train Time (s)': r['train_time']
    } for name, r in results.items()
}).T.round(4)

print("=== Model Comparison ===")
print(comparison_df.to_string())

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.2

for i, (name, r) in enumerate(results.items()):
    values = [r['precision'], r['recall'], r['f1'], r['roc_auc']]
    axes[0].bar(x + i*width, values, width, label=name, edgecolor='black')

axes[0].set_xticks(x + width*(len(results)-1)/2)
axes[0].set_xticklabels(metrics)
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.1)

# ROC-AUC bar chart
model_names = list(results.keys())
roc_aucs = [results[n]['roc_auc'] for n in model_names]
colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12'][:len(model_names)]
axes[1].bar(model_names, roc_aucs, color=colors_bar, edgecolor='black')
axes[1].set_ylabel('ROC-AUC Score')
axes[1].set_title('ROC-AUC Score by Model')
axes[1].set_ylim(0.5, 1.0)
for i, v in enumerate(roc_aucs):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 5.4 Hyperparameter Tuning (Best Model)

We select the best performing model and tune its hyperparameters using RandomizedSearchCV for efficiency.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# Select the best model based on F1-Score
best_model_name = max(results, key=lambda x: results[x]['f1'])
print(f"Best model so far: {best_model_name} (F1: {results[best_model_name]['f1']:.4f})")

# Hyperparameter tuning for Random Forest (commonly best for this type)
rf_param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Use stratified k-fold on a smaller sample for tuning
tune_size = min(30000, len(X_train_smote))
tune_idx = np.random.choice(len(X_train_smote), size=tune_size, replace=False)
X_tune = X_train_smote[tune_idx] if isinstance(X_train_smote, np.ndarray) else X_train_smote.iloc[tune_idx]
y_tune = y_train_smote[tune_idx] if isinstance(y_train_smote, np.ndarray) else y_train_smote.iloc[tune_idx]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rf_tuner = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=10,  # Number of parameter settings sampled
    cv=cv,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nRunning RandomizedSearchCV for Random Forest...")
rf_tuner.fit(X_tune, y_tune)

print(f"\nBest Parameters: {rf_tuner.best_params_}")
print(f"Best CV F1-Score: {rf_tuner.best_score_:.4f}")

# Evaluate tuned model on test set
y_pred_tuned = rf_tuner.best_estimator_.predict(X_test_scaled)
y_pred_proba_tuned = rf_tuner.best_estimator_.predict_proba(X_test_scaled)[:, 1]

print(f"\n=== Tuned Random Forest on Test Set ===")
print(classification_report(y_test, y_pred_tuned, target_names=['Legitimate', 'Fraudulent']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_tuned):.4f}")

---
# Task 6: ANN Model Development

We design and train an Artificial Neural Network using TensorFlow/Keras.

### Architecture Design:
- **Input Layer:** 15 features (matching our feature set)
- **Hidden Layer 1:** 64 neurons, ReLU activation, BatchNormalization, Dropout(0.3)
- **Hidden Layer 2:** 32 neurons, ReLU activation, BatchNormalization, Dropout(0.3)
- **Hidden Layer 3:** 16 neurons, ReLU activation, Dropout(0.2)
- **Output Layer:** 1 neuron, Sigmoid activation (binary classification)

### Justification:
- **ReLU activation:** Prevents vanishing gradient, computationally efficient
- **BatchNormalization:** Stabilizes learning, allows higher learning rates
- **Dropout:** Prevents overfitting
- **Adam optimizer:** Adaptive learning rate, works well for most problems
- **Binary crossentropy loss:** Standard for binary classification
- **Learning rate = 0.001:** Good default for Adam optimizer

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF warnings

print(f"TensorFlow version: {tf.__version__}")

# Build ANN model
def build_ann(input_dim, learning_rate=0.001):
    model = keras.Sequential([
        # Input layer
        layers.Input(shape=(input_dim,)),
        
        # Hidden Layer 1
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Hidden Layer 2
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Hidden Layer 3
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.2),
        
        # Output Layer
        layers.Dense(1, activation='sigmoid')
    ])
    
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, 
                  loss='binary_crossentropy',
                  metrics=['accuracy', keras.metrics.AUC(name='auc'),
                           keras.metrics.Precision(name='precision'),
                           keras.metrics.Recall(name='recall')])
    return model

ann_model = build_ann(input_dim=X_train_smote.shape[1])
ann_model.summary()

In [ ]:
# Training the ANN
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Convert to numpy arrays if needed
X_train_ann = np.array(X_train_smote)
y_train_ann = np.array(y_train_smote)
X_test_ann = np.array(X_test_scaled)
y_test_ann = np.array(y_test)

print("Training ANN model...")
history = ann_model.fit(
    X_train_ann, y_train_ann,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
print("\nTraining complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0, 1].set_title('Training & Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()

# AUC
axes[1, 0].plot(history.history['auc'], label='Train AUC', linewidth=2)
axes[1, 0].plot(history.history['val_auc'], label='Val AUC', linewidth=2)
axes[1, 0].set_title('Training & Validation AUC')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUC')
axes[1, 0].legend()

# Precision & Recall
axes[1, 1].plot(history.history['precision'], label='Train Precision', linewidth=2)
axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(history.history['val_precision'], label='Val Precision', linewidth=2, linestyle='--')
axes[1, 1].plot(history.history['val_recall'], label='Val Recall', linewidth=2, linestyle='--')
axes[1, 1].set_title('Precision & Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend(fontsize=8)

plt.suptitle('ANN Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate ANN on test set
y_pred_ann_proba = ann_model.predict(X_test_ann).flatten()
y_pred_ann = (y_pred_ann_proba >= 0.5).astype(int)

ann_precision = precision_score(y_test, y_pred_ann)
ann_recall = recall_score(y_test, y_pred_ann)
ann_f1 = f1_score(y_test, y_pred_ann)
ann_roc_auc = roc_auc_score(y_test, y_pred_ann_proba)

print("=== ANN Model Results on Test Set ===")
print(classification_report(y_test, y_pred_ann, target_names=['Legitimate', 'Fraudulent']))
print(f"ROC-AUC: {ann_roc_auc:.4f}")

# Add ANN to results
results['ANN'] = {
    'precision': ann_precision,
    'recall': ann_recall,
    'f1': ann_f1,
    'roc_auc': ann_roc_auc,
    'y_pred': y_pred_ann,
    'y_pred_proba': y_pred_ann_proba,
    'train_time': 0  # Not timed
}

---
## Improved ANN Model (V2)

Let's build a better ANN with:
- Deeper architecture (128-64-32-16)
- Class weights to handle imbalance better
- Lower initial learning rate
- More aggressive regularization

In [ ]:
old_ann_results = {
    'precision': results['ANN']['precision'],
    'recall': results['ANN']['recall'],
    'f1': results['ANN']['f1'],
    'roc_auc': results['ANN']['roc_auc'],
    'accuracy': results['ANN']['accuracy']
}
print("Old ANN scores saved for comparison")

In [ ]:
def build_improved_ann(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

ann_v2 = build_improved_ann(X_train_smote.shape[1])
ann_v2.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train_smote), y=y_train_smote)
class_weight_dict = dict(enumerate(class_weights))

history_v2 = ann_v2.fit(
    np.array(X_train_smote), np.array(y_train_smote),
    epochs=80, batch_size=512, validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', patience=8, restore_best_weights=True, mode='max'),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7)
    ], verbose=1
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].plot(history_v2.history['loss'], label='Train'); axes[0,0].plot(history_v2.history['val_loss'], label='Val')
axes[0,0].set_title('Loss (V2)'); axes[0,0].legend()
axes[0,1].plot(history_v2.history['accuracy'], label='Train'); axes[0,1].plot(history_v2.history['val_accuracy'], label='Val')
axes[0,1].set_title('Accuracy (V2)'); axes[0,1].legend()
axes[1,0].plot(history_v2.history['auc'], label='Train'); axes[1,0].plot(history_v2.history['val_auc'], label='Val')
axes[1,0].set_title('AUC (V2)'); axes[1,0].legend()
axes[1,1].plot(history_v2.history['precision'], label='Prec')
axes[1,1].plot(history_v2.history['recall'], label='Rec')
axes[1,1].plot(history_v2.history['val_precision'], label='Val Prec', linestyle='--')
axes[1,1].plot(history_v2.history['val_recall'], label='Val Rec', linestyle='--')
axes[1,1].set_title('Precision & Recall (V2)'); axes[1,1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
y_v2_proba = ann_v2.predict(np.array(X_test_processed)).flatten()
y_v2_pred = (y_v2_proba >= 0.5).astype(int)

new_ann_results = {
    'accuracy': accuracy_score(y_test, y_v2_pred),
    'precision': precision_score(y_test, y_v2_pred),
    'recall': recall_score(y_test, y_v2_pred),
    'f1': f1_score(y_test, y_v2_pred),
    'roc_auc': roc_auc_score(y_test, y_v2_proba)
}

print(classification_report(y_test, y_v2_pred, target_names=['Legitimate','Fraudulent']))

### ANN V1 vs V2 Comparison

In [ ]:
comparison = pd.DataFrame({
    'ANN V1 (Old)': old_ann_results,
    'ANN V2 (Improved)': new_ann_results
}).round(4)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = list(old_ann_results.keys())
v1_vals = [old_ann_results[m] for m in metrics]
v2_vals = [new_ann_results[m] for m in metrics]
x = np.arange(len(metrics))

axes[0].bar(x - 0.15, v1_vals, 0.3, label='ANN V1')
axes[0].bar(x + 0.15, v2_vals, 0.3, label='ANN V2')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=15)
axes[0].set_title('ANN V1 vs V2'); axes[0].legend()
axes[0].set_ylim(0, 1.1)

fpr1, tpr1, _ = roc_curve(y_test, results['ANN']['y_pred_proba'])
fpr2, tpr2, _ = roc_curve(y_test, y_v2_proba)
axes[1].plot(fpr1, tpr1, label=f'V1 (AUC={old_ann_results["roc_auc"]:.4f})')
axes[1].plot(fpr2, tpr2, label=f'V2 (AUC={new_ann_results["roc_auc"]:.4f})')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curve Comparison'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
results['ANN V2'] = {
    'accuracy': new_ann_results['accuracy'],
    'precision': new_ann_results['precision'],
    'recall': new_ann_results['recall'],
    'f1': new_ann_results['f1'],
    'roc_auc': new_ann_results['roc_auc'],
    'y_pred': y_v2_pred,
    'y_pred_proba': y_v2_proba
}

ann_v2.save('best_ann_model_v2.keras')

---
# Task 7: Model Evaluation

Comprehensive evaluation using Precision, Recall, F1-Score, and ROC-AUC. We compare ML models vs ANN.

In [ ]:
# Confusion Matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, r) in enumerate(results.items()):
    if idx >= len(axes):
        break
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    axes[idx].set_title(f'{name}\nF1: {r["f1"]:.4f} | AUC: {r["roc_auc"]:.4f}')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

# Hide unused subplots
for idx in range(len(results), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
for i, (name, r) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, r['y_pred_proba'])
    axes[0].plot(fpr, tpr, color=colors[i % len(colors)], linewidth=2,
                 label=f'{name} (AUC={r["roc_auc"]:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves - All Models')
axes[0].legend(fontsize=9)

# Precision-Recall Curves
for i, (name, r) in enumerate(results.items()):
    prec, rec, _ = precision_recall_curve(y_test, r['y_pred_proba'])
    axes[1].plot(rec, prec, color=colors[i % len(colors)], linewidth=2, label=name)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves - All Models')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### 7.1 ML vs ANN Comparison

In [ ]:
# Final comparison table
all_results = pd.DataFrame({
    name: {
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1-Score': r['f1'],
        'ROC-AUC': r['roc_auc']
    } for name, r in results.items()
}).T.round(4)

print("="*60)
print("       FINAL MODEL COMPARISON (ML vs ANN)")
print("="*60)
print(all_results.to_string())
print("\n" + "="*60)

# Highlight best
best_f1_model = all_results['F1-Score'].idxmax()
best_auc_model = all_results['ROC-AUC'].idxmax()
print(f"\n Best F1-Score: {best_f1_model} ({all_results.loc[best_f1_model, 'F1-Score']:.4f})")
print(f" Best ROC-AUC: {best_auc_model} ({all_results.loc[best_auc_model, 'ROC-AUC']:.4f})")

---
# Task 8: Hyperparameter Tuning & Cross Validation

### 8.1 Stratified K-Fold Cross Validation

For imbalanced datasets, stratified k-fold is essential to maintain class proportions in each fold.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Cross-validation on the best ML model using SMOTE data
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use smaller sample for CV to be feasible
cv_size = min(20000, len(X_train_smote))
cv_idx = np.random.choice(len(X_train_smote), size=cv_size, replace=False)
X_cv = X_train_smote[cv_idx] if isinstance(X_train_smote, np.ndarray) else X_train_smote.iloc[cv_idx]
y_cv = y_train_smote[cv_idx] if isinstance(y_train_smote, np.ndarray) else y_train_smote.iloc[cv_idx]

print("=== 5-Fold Stratified Cross Validation ===\n")

for name in ['Logistic Regression', 'Random Forest']:
    if name in models:
        model_cv = models[name].__class__(**models[name].get_params())
        
        # F1 score CV
        f1_scores = cross_val_score(model_cv, X_cv, y_cv, cv=cv_stratified, scoring='f1', n_jobs=-1)
        roc_scores = cross_val_score(model_cv, X_cv, y_cv, cv=cv_stratified, scoring='roc_auc', n_jobs=-1)
        
        print(f"{name}:")
        print(f"  F1-Score:  {f1_scores.mean():.4f} (+/- {f1_scores.std()*2:.4f})")
        print(f"  ROC-AUC:   {roc_scores.mean():.4f} (+/- {roc_scores.std()*2:.4f})")
        print(f"  Individual folds (F1): {[f'{s:.4f}' for s in f1_scores]}")
        print()

### 8.2 GridSearchCV for Fine-Tuning

We use GridSearchCV with a focused parameter grid for the top-performing model.

In [ ]:
from sklearn.model_selection import GridSearchCV

# GridSearchCV for Logistic Regression (quick to tune)
lr_param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'max_iter': [1000]
}

print("Running GridSearchCV for Logistic Regression...")
lr_grid = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid=lr_param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_cv, y_cv)

print(f"\nBest Parameters: {lr_grid.best_params_}")
print(f"Best CV F1-Score: {lr_grid.best_score_:.4f}")

# Test set evaluation
y_pred_grid = lr_grid.best_estimator_.predict(X_test_scaled)
y_pred_proba_grid = lr_grid.best_estimator_.predict_proba(X_test_scaled)[:, 1]
print(f"\nTest Set Performance:")
print(f"  F1-Score: {f1_score(y_test, y_pred_grid):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_grid):.4f}")

---
# Task 9: Overfitting / Underfitting Analysis

We compare training vs validation performance to check for overfitting.

In [ ]:
# Training vs Test performance comparison
print("=== Overfitting/Underfitting Analysis ===\n")

for name in ['Logistic Regression', 'Random Forest']:
    if name in models:
        model_check = models[name]
        
        # Training performance
        y_train_pred = model_check.predict(X_train_smote)
        train_f1 = f1_score(y_train_smote, y_train_pred)
        train_acc = (y_train_pred == y_train_smote).mean()
        
        # Test performance
        test_f1 = results[name]['f1']
        test_acc = (results[name]['y_pred'] == np.array(y_test)).mean()
        
        gap_f1 = train_f1 - test_f1
        gap_acc = train_acc - test_acc
        
        print(f"{name}:")
        print(f"  Train Accuracy: {train_acc:.4f} | Test Accuracy: {test_acc:.4f} | Gap: {gap_acc:.4f}")
        print(f"  Train F1-Score: {train_f1:.4f} | Test F1-Score: {test_f1:.4f} | Gap: {gap_f1:.4f}")
        
        if gap_f1 > 0.1:
            print(f"  [!] Possible OVERFITTING (F1 gap > 0.1)")
        elif gap_f1 < -0.05:
            print(f"  [!] Possible UNDERFITTING")
        else:
            print(f"  [OK] Good generalization")
        print()

# ANN analysis from training history
print("ANN Model:")
if 'val_loss' in history.history:
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    final_train_auc = history.history['auc'][-1]
    final_val_auc = history.history['val_auc'][-1]
    
    print(f"  Train Loss: {final_train_loss:.4f} | Val Loss: {final_val_loss:.4f}")
    print(f"  Train AUC: {final_train_auc:.4f} | Val AUC: {final_val_auc:.4f}")
    
    if final_val_loss > final_train_loss * 1.3:
        print(f"  [!] Some overfitting detected")
    else:
        print(f"  [OK] Reasonable generalization")

### 9.1 Suggestions for Improvement

1. **More Data:** Larger training samples could improve generalization
2. **Feature Engineering:** More sophisticated features (e.g., transaction velocity, spending patterns)
3. **Ensemble Methods:** Combining multiple models for better predictions
4. **Threshold Tuning:** Adjusting the classification threshold for optimal precision-recall tradeoff
5. **Additional Regularization:** For ANN, try different dropout rates
6. **Cost-sensitive Learning:** Assign higher misclassification costs to fraud cases

---
# Task 10: Final Model Justification & Selection

In [ ]:
# Final comprehensive comparison
print("="*65)
print("          FINAL MODEL SELECTION & JUSTIFICATION")
print("="*65)

# Get all results
final_comparison = pd.DataFrame({
    name: {
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1-Score': r['f1'],
        'ROC-AUC': r['roc_auc']
    } for name, r in results.items()
}).T

print("\n--- Performance Summary ---")
print(final_comparison.round(4).to_string())

# Determine best model
best_model_name = final_comparison['F1-Score'].idxmax()
best_roc_model = final_comparison['ROC-AUC'].idxmax()

print(f"\n{'='*65}")
print(f"[BEST] BEST MODEL: {best_model_name}")
print(f"{'='*65}")
print(f"\nSelected based on:")
print(f"  * Highest F1-Score: {final_comparison.loc[best_model_name, 'F1-Score']:.4f}")
print(f"  * ROC-AUC: {final_comparison.loc[best_model_name, 'ROC-AUC']:.4f}")
print(f"  * Good balance between Precision and Recall")
print(f"\nJustification:")
print(f"  1. For fraud detection, we need high Recall (catch as many frauds as possible)")
print(f"  2. F1-Score balances Precision and Recall, making it the best single metric")
print(f"  3. ROC-AUC shows the model's ability to discriminate between classes")
print(f"  4. The selected model generalizes well (small train-test gap)")

In [ ]:
# Save the best model for deployment
import joblib

# Save the best ML model
best_ml_name = max(
    {k: v for k, v in results.items() if k != 'ANN'}, 
    key=lambda x: results[x]['f1']
)

if best_ml_name in models:
    joblib.dump(models[best_ml_name], 'best_ml_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')
    joblib.dump(le_cat, 'label_encoder_category.pkl')
    joblib.dump(le_gen, 'label_encoder_gender.pkl')
    print(f"Saved best ML model ({best_ml_name}) as 'best_ml_model.pkl'")
    print(f"Saved scaler as 'scaler.pkl'")
    print(f"Saved label encoders")

# Save ANN model  
ann_model.save('best_ann_model.keras')
print(f"Saved ANN model as 'best_ann_model.keras'")

print(f"\n[OK] Models saved and ready for deployment!")

---
# Summary

## What we accomplished:

| Task | Description | Status |
|------|-----------|--------|
| 1 | Data Understanding & Cleaning | [OK] |
| 2 | Exploratory Data Analysis (EDA) | [OK] |
| 3 | Data Preprocessing (Scaling, SMOTE, Split) | [OK] |
| 4 | Feature Engineering & Selection (Mutual Information) | [OK] |
| 5 | Model Building (4+ ML Models) | [OK] |
| 6 | ANN Model Development | [OK] |
| 7 | Model Evaluation (Precision, Recall, F1, ROC-AUC) | [OK] |
| 8 | Hyperparameter Tuning & Cross Validation | [OK] |
| 9 | Overfitting/Underfitting Analysis | [OK] |
| 10 | Final Model Justification | [OK] |

## Key Findings:
- The dataset is **highly imbalanced** (~0.58% fraud rate)
- **SMOTE** was used to handle class imbalance (applied only on training data)
- **Feature engineering** (distance, time features, log-transform) significantly helped
- **Random Forest / XGBoost** typically perform best on this type of tabular data
- **ANN** also shows competitive performance
- **F1-Score and ROC-AUC** are the most meaningful metrics for this imbalanced problem
- Models are saved and ready for deployment!